In [1]:
#### Pydantic is useful with FastAPI

In [9]:
pip install pydantic

Note: you may need to restart the kernel to use updated packages.


In [10]:
pip install pydantic[email]

Note: you may need to restart the kernel to use updated packages.


In [11]:
pip install pydantic-settings

Note: you may need to restart the kernel to use updated packages.


In [126]:
from datetime import date
from uuid import UUID, uuid4
from enum import Enum
from pydantic import BaseModel, EmailStr, Field

class Department(Enum):
    HR = "HR"
    SALES = "SALES"
    IT = "IT"
    ENGINEERING = "ENG"

class Employee(BaseModel):
    employee_id: UUID = Field(default_factory=uuid4, frozen=True)               # "default factory" a callable that generates default values
    name: str = Field(min_length=1, frozen=True)                                # "frozen" Boolean parameter to make your fields immutable
                                                                                # control the length of string fields with min_length and max_length
    email: EmailStr = Field(pattern=r".+@example\.com$")                        # you can set pattern to a regex expression
    date_of_birth: date = Field(alias="birth_date", repr=False, frozen=True)    # "alias" parameter when you want to assign an alias to your fields
    salary: float = Field(alias="compensation", gt=0, repr=True)               # "gt" greater than, pydantic has gt,lt,ge,le,multiple_of, allow_inf_nan
                                                                                # "repr" whether a field is displayed in the model’s field representation
    department: Department
    elected_benefits: bool

In [128]:
employee_data=Employee(
    name="Chris DeTuma",
    email="cdetuma@example.com",
    birth_date="1998-04-02",
    compensation=123_000.00,
    department="ENG",
    elected_benefits=True,
)

employee = Employee.model_validate(employee_data)
employee

# Because you set repr=False, date_of_birth is not displayed in the Employee representation

Employee(employee_id=UUID('09e69c75-98ba-4335-9f13-145b83b478d7'), name='Chris DeTuma', email='cdetuma@example.com', salary=123000.0, department=<Department.ENGINEERING: 'ENG'>, elected_benefits=True)

In [129]:
employee.date_of_birth

datetime.date(1998, 4, 2)

In [130]:
employee.department = "HR"
employee.name = "Andrew TuGrendele"

ValidationError: 1 validation error for Employee
name
  Field is frozen [type=frozen_field, input_value='Andrew TuGrendele', input_type=str]
    For further information visit https://errors.pydantic.dev/2.8/v/frozen_field

In [131]:
employee

Employee(employee_id=UUID('09e69c75-98ba-4335-9f13-145b83b478d7'), name='Chris DeTuma', email='cdetuma@example.com', salary=123000.0, department='HR', elected_benefits=True)

#### Instantiate employee from dictionary

In [136]:
new_employee_dict = {
    "name": "Chris DeTuma",
    "email": "cdetuma@example.com",
    "birth_date": "1998-04-02",
    "compensation": 123_000.00,
    "department": "IT",
    "elected_benefits": True,
}

dict_employee=Employee.model_validate(new_employee_dict)
dict_employee

Employee(employee_id=UUID('d08696ab-07a1-4913-bcba-537f42073d44'), name='Chris DeTuma', email='cdetuma@example.com', salary=123000.0, department=<Department.IT: 'IT'>, elected_benefits=True)

In [137]:
dict_employee.model_dump()

{'employee_id': UUID('d08696ab-07a1-4913-bcba-537f42073d44'),
 'name': 'Chris DeTuma',
 'email': 'cdetuma@example.com',
 'date_of_birth': datetime.date(1998, 4, 2),
 'salary': 123000.0,
 'department': <Department.IT: 'IT'>,
 'elected_benefits': True}

#### Instantiate employee from JSON

In [139]:
new_employee_json = """
 {"employee_id":"d2e7b773-926b-49df-939a-5e98cbb9c8eb",
  "name":"Eric Slogren",
  "email":"eslogrenta@example.com",
  "birth_date":"1990-01-02",
  "compensation":125000.0,
  "department":"ENG",
  "elected_benefits":false}
  """

json_employee=Employee.model_validate_json(new_employee_json)
json_employee

Employee(employee_id=UUID('d2e7b773-926b-49df-939a-5e98cbb9c8eb'), name='Eric Slogren', email='eslogrenta@example.com', salary=125000.0, department=<Department.ENGINEERING: 'ENG'>, elected_benefits=False)

In [140]:
json_employee.model_dump_json()
#DOB and Dept converted to string

'{"employee_id":"d2e7b773-926b-49df-939a-5e98cbb9c8eb","name":"Eric Slogren","email":"eslogrenta@example.com","date_of_birth":"1990-01-02","salary":125000.0,"department":"ENG","elected_benefits":false}'

#### Field Validators

In [149]:
from datetime import date
from uuid import UUID, uuid4
from enum import Enum
from pydantic import BaseModel, EmailStr, Field, field_validator

class Department(Enum):
    HR = "HR"
    SALES = "SALES"
    IT = "IT"
    ENGINEERING = "ENG"

class Employee(BaseModel):
    employee_id: UUID = Field(default_factory=uuid4, frozen=True)
    name: str = Field(min_length=1, frozen=True)
    email: EmailStr = Field(pattern=r".+@example\.com$")
    date_of_birth: date = Field(alias="birth_date", repr=False, frozen=True)
    salary: float = Field(alias="compensation", gt=0, repr=False)
    department: Department
    elected_benefits: bool

    @field_validator("date_of_birth")
    @classmethod
    def check_valid_age(cls, date_of_birth: date) -> date:
        today = date.today()
        eighteen_years_ago = date(today.year - 18, today.month, today.day)

        if date_of_birth > eighteen_years_ago:
            raise ValueError("Employees must be at least 18 years old.")

        return date_of_birth

In [153]:
employee_data=Employee(
    name="Chris DeTuma",
    email="cdetuma@example.com",
    birth_date="2008-04-02",
    compensation=123_000.00,
    department="ENG",
    elected_benefits=True,
)

employee = Employee.model_validate(employee_data)
employee

ValidationError: 1 validation error for Employee
birth_date
  Value error, Employees must be at least 18 years old. [type=value_error, input_value='2008-04-02', input_type=str]
    For further information visit https://errors.pydantic.dev/2.8/v/value_error

#### Model Validators

In [154]:
from typing import Self
from datetime import date
from uuid import UUID, uuid4
from enum import Enum
from pydantic import (
    BaseModel,
    EmailStr,
    Field,
    field_validator,
    model_validator,
)

class Department(Enum):
    HR = "HR"
    SALES = "SALES"
    IT = "IT"
    ENGINEERING = "ENGINEERING"

class Employee(BaseModel):
    employee_id: UUID = Field(default_factory=uuid4, frozen=True)
    name: str = Field(min_length=1, frozen=True)
    email: EmailStr = Field(pattern=r".+@example\.com$")
    date_of_birth: date = Field(alias="birth_date", repr=False, frozen=True)
    salary: float = Field(alias="compensation", gt=0, repr=False)
    department: Department
    elected_benefits: bool

    @field_validator("date_of_birth")
    @classmethod
    def check_valid_age(cls, date_of_birth: date) -> date:
        today = date.today()
        eighteen_years_ago = date(today.year - 18, today.month, today.day)

        if date_of_birth > eighteen_years_ago:
            raise ValueError("Employees must be at least 18 years old.")

        return date_of_birth

    @model_validator(mode="after")                                        # mode=after, Pydantic waits until after you’ve instantiated your model to run
    def check_it_benefits(self) -> Self:
        department = self.department
        elected_benefits = self.elected_benefits

        if department == Department.IT and elected_benefits:
            raise ValueError(
                "IT employees are contractors and don't qualify for benefits"
            )
        return self

In [156]:
new_employee_data=Employee(
    name="Chris DeTuma",
    email="cdetuma@example.com",
    birth_date="1998-04-02",
    compensation=123_000.00,
    department="IT",
    elected_benefits=True,
)

Employee.model_validate(new_employee_data)

ValidationError: 1 validation error for Employee
  Value error, IT employees are contractors and don't qualify for benefits [type=value_error, input_value={'name': 'Chris DeTuma', ...elected_benefits': True}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.8/v/value_error

#### BaseSettings

In [161]:
from pydantic import HttpUrl, Field
from pydantic_settings import BaseSettings

class AppConfig(BaseSettings):
    database_host: HttpUrl
    database_user: str = Field(min_length=5)
    database_password: str = Field(min_length=10)
    api_key: str = Field(min_length=20)

In [162]:
AppConfig()

ValidationError: 4 validation errors for AppConfig
database_host
  Field required [type=missing, input_value={}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.8/v/missing
database_user
  Field required [type=missing, input_value={}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.8/v/missing
database_password
  Field required [type=missing, input_value={}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.8/v/missing
api_key
  Field required [type=missing, input_value={}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.8/v/missing

#### Customizing BaseSettings With SettingsConfigDict

In [168]:
from pydantic import HttpUrl, Field
from pydantic_settings import BaseSettings, SettingsConfigDict

class AppConfig(BaseSettings):
    model_config = SettingsConfigDict(
        env_file=".env",
        env_file_encoding="utf-8",
        case_sensitive=True,
        extra="forbid",
    )

    database_host: HttpUrl
    database_user: str = Field(min_length=5)
    database_password: str = Field(min_length=10)
    api_key: str = Field(min_length=20)

In [169]:
AppConfig()

ValidationError: 4 validation errors for AppConfig
database_host
  Field required [type=missing, input_value={}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.8/v/missing
database_user
  Field required [type=missing, input_value={}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.8/v/missing
database_password
  Field required [type=missing, input_value={}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.8/v/missing
api_key
  Field required [type=missing, input_value={}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.8/v/missing